# Algorithm Comparison
**Purpose:** Compare KMeans, HDBSCAN and Gaussian Mixture on the same data and choose the final regime solution.

**Inputs:** `data/results/kmeans_labels.csv`, `gmm_labels.csv`, `hdbscan_mcs20_ms10_labels.csv`, scaled features.

**Outputs:** `data/results/method_comparison.csv`, `method_agreement.csv`

## 1. Load all results

In [2]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

BASE = Path.cwd().parent
X = pd.read_csv(BASE / "data" / "features" / "delhi_scaled.csv", parse_dates=["Date"], index_col="Date")
Xn = X.drop(columns=["wind_dir_sin", "wind_dir_cos"])
Xs = Xn.drop(columns=["SO2"])
feat = pd.read_csv(BASE / "data" / "features" / "delhi_features.csv", parse_dates=["Date"], index_col="Date")
ctx = pd.read_csv(BASE / "data" / "features" / "delhi_context.csv", parse_dates=["Date"], index_col="Date")
raw = feat.drop(columns=["wind_dir_sin", "wind_dir_cos"])

res_dir = BASE / "data" / "results"
def load(name):
    return pd.read_csv(res_dir / name, parse_dates=["Date"], index_col="Date").iloc[:, 0]

km = load("kmeans_labels.csv")
gmm = load("gmm_labels.csv")
try:
    hdb = load("hdbscan_mcs20_ms10_labels.csv")
except FileNotFoundError:      # notebook 07's save cell was not run
    hdb = pd.Series(HDBSCAN(min_cluster_size=20, min_samples=10, copy=True).fit_predict(Xn), index=Xn.index)
print(Xs.shape, len(km), len(gmm), len(hdb))     

(1195, 12) 1195 1195 1195


## 2. KMeans without SO2 (like-for-like with the GMM)

In [3]:
km_s = pd.Series(KMeans(n_clusters=5, n_init=20, random_state=42).fit_predict(Xs), index=Xs.index)
print("ARI, KMeans with SO2 vs without:", round(adjusted_rand_score(km, km_s), 3))

ARI, KMeans with SO2 vs without: 0.843


## 3. Metrics table
Silhouette and Davies-Bouldin are computed in each method's own feature space, and HDBSCAN's only on non-noise days, so they are indicative, not strictly comparable.

In [4]:
def stab_km(data, k=5, n=10):
    base = KMeans(n_clusters=k, n_init=1, random_state=0).fit_predict(data)
    return np.mean([adjusted_rand_score(base, KMeans(n_clusters=k, n_init=1, random_state=s).fit_predict(data)) for s in range(1, n + 1)])

def stab_gm(data, k=5, n=10):
    base = GaussianMixture(n_components=k, covariance_type="full", random_state=0).fit(data).predict(data)
    return np.mean([adjusted_rand_score(base, GaussianMixture(n_components=k, covariance_type="full", random_state=s).fit(data).predict(data)) for s in range(1, n + 1)])

methods = {   # name: (labels, feature space the method was fit on, stability)
    "KMeans k=5 (13 features)": (km, Xn, stab_km(Xn)),
    "KMeans k=5 (no SO2)":      (km_s, Xs, stab_km(Xs)),
    "GMM k=5 (no SO2)":         (gmm, Xs, stab_gm(Xs)),
    "HDBSCAN mcs20/ms10":       (hdb, Xn, np.nan),
}

rows = []
for name, (lab, data, st) in methods.items():
    keep = lab != -1
    n_cl = lab[keep].nunique()
    rows.append({
        "method": name,
        "clusters": n_cl,
        "noise_%": round((~keep).mean() * 100, 1),
        "largest_cluster_%": round(lab[keep].value_counts(normalize=True).max() * 100, 1),
        "silhouette": round(silhouette_score(data[keep], lab[keep]), 3) if n_cl > 1 else np.nan,
        "davies_bouldin": round(davies_bouldin_score(data[keep], lab[keep]), 3) if n_cl > 1 else np.nan,
        "stability": round(st, 3),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

                  method  clusters  noise_%  largest_cluster_%  silhouette  davies_bouldin  stability
KMeans k=5 (13 features)         5      0.0               25.0       0.203           1.585      0.974
     KMeans k=5 (no SO2)         5      0.0               24.0       0.208           1.567      0.900
        GMM k=5 (no SO2)         5      0.0               33.0       0.083           2.847      0.620
      HDBSCAN mcs20/ms10         3     74.7               79.8       0.107           1.359        NaN


## 4. Agreement between methods (ARI)

In [5]:
labs = {"KMeans": km, "KMeans noSO2": km_s, "GMM": gmm, "HDBSCAN": hdb}
names = list(labs)
M = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        keep = (labs[a] != -1) & (labs[b] != -1)      # HDBSCAN noise days are left out
        M.loc[a, b] = adjusted_rand_score(labs[a][keep], labs[b][keep])
print(M.round(2))

              KMeans  KMeans noSO2   GMM  HDBSCAN
KMeans          1.00          0.84  0.25     0.13
KMeans noSO2    0.84          1.00  0.28     0.11
GMM             0.25          0.28  1.00     0.20
HDBSCAN         0.13          0.11  0.20     1.00


## 5. Which regimes do the methods find in common?
Best-matching group for each KMeans regime (overlap 1.0 = identical).

In [6]:
def best_match(ref, other):
    out = []
    for r in sorted(ref.unique()):
        a = set(ref.index[ref == r])
        best = max(((o, len(a & set(other.index[other == o])) / len(a | set(other.index[other == o])))
                    for o in other.unique() if o != -1), key=lambda t: t[1])
        out.append((r, len(a), best[0], round(best[1], 2)))
    return pd.DataFrame(out, columns=["kmeans_cluster", "days", "best_match", "overlap"])

names_km = {0: "monsoon washout", 1: "moderate winter smog", 2: "pre-monsoon dust",
            3: "severe stagnant winter smog", 4: "warm, cleaner days"}
for nm, lab in [("GMM", gmm), ("HDBSCAN", hdb)]:
    t = best_match(km, lab)
    t["regime"] = t["kmeans_cluster"].map(names_km)
    print(f"\nBest match for each KMeans regime in {nm} (overlap 1.0 = identical):")
    print(t.to_string(index=False))


Best match for each KMeans regime in GMM (overlap 1.0 = identical):
 kmeans_cluster  days  best_match  overlap                      regime
              0   175           3     0.66             monsoon washout
              1   299           2     0.30        moderate winter smog
              2   219           4     0.32            pre-monsoon dust
              3   231           2     0.41 severe stagnant winter smog
              4   271           4     0.28          warm, cleaner days

Best match for each KMeans regime in HDBSCAN (overlap 1.0 = identical):
 kmeans_cluster  days  best_match  overlap                      regime
              0   175           2     0.16             monsoon washout
              1   299           1     0.22        moderate winter smog
              2   219           1     0.06            pre-monsoon dust
              3   231           1     0.23 severe stagnant winter smog
              4   271           1     0.06          warm, cleaner days


## 6. Save comparison tables

In [8]:
table.to_csv(res_dir / "method_comparison.csv", index=False)
M.round(3).to_csv(res_dir / "method_agreement.csv")